# eph_02 — Monovariate kinematics encoding

Does spike count predict tongue kinematics?

**Pipeline:**
1. Data loading via `data_loading.py`
2. Trial × unit table via `ephys_utils.build_all_counts_df`
3. `AnalysisSpec(method="spearman")` + `fit_encoding` — per-unit Spearman for each kinematic predictor
4. `PerUnitStatsRegistry` — standard output for downstream comparison (eph_03)
5. Diagnostics: T distributions, pairwise significance UpSet + heatmap, example units, session-level RT~kin correlations

## 1. Setup

In [ ]:
%matplotlib inline

import contextlib, io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
from scipy.stats import spearmanr

from plotstyle import apply_style, PALETTE, OKABE_ITO, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_02_kin_encoding"
SAVE_FIG = False

print(f"ENV       : {ENV}")
print(f"FOR_LOCAL : {FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter,
    filter_ephys_units,
    load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)

    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)

    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

## 3. Build trial × unit table

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)

if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")

# Derive absolute excursion angle (sign encodes direction, not magnitude)
all_counts_df["first_move_excursion_angle_deg_abs"] = (
    all_counts_df["first_move_excursion_angle_deg"].abs()
)

print("all_counts_df shape:", all_counts_df.shape)

## 4. Analysis specifications

One `AnalysisSpec(method="spearman")` per kinematic feature. `spike_count` is the
predictor and the kinematic variable is the response — this asks whether
spiking predicts subsequent movement parameters.

In [ ]:
from encoding_methods import AnalysisSpec, AnalysisResult, fit_encoding
from encoding_plots import (tstat_hist, t_scatter, registry_heatmap,
                             registry_upset, registry_plot_examples)

# Kinematic predictors to test
KIN_PREDICTORS = {
    "endpoint_x":  "first_move_endpoint_x",
    "endpoint_y":  "first_move_endpoint_y",
    "angle":       "first_move_excursion_angle_deg_abs",
    "peak_vel":    "first_move_peak_velocity",
    "mean_vel":    "first_move_out_mean_velocity",
    "out_peak_v":  "first_move_out_peak_velocity",
    "duration":    "first_move_out_duration",
    "distance":    "first_move_out_total_distance",
}

# Loose filter: require valid RT but don't restrict range (kinematics analysis)
KIN_QUERY = "reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.0"

specs = [
    AnalysisSpec(
        name=f"spearman_{key}",
        predictor_col="spike_count",
        response_col=col,
        method="spearman",
        trial_query=KIN_QUERY,
        notes=f"Spearman: spike_count vs {col}",
    )
    for key, col in KIN_PREDICTORS.items()
]

## 5. Run encoding analyses

In [ ]:
results = {}
for spec in specs:
    results[spec.name] = fit_encoding(all_counts_df, spec)

## 6. Register results

In [ ]:
from per_unit_stats_registry import PerUnitStatsRegistry
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)

for spec in specs:
    reg.register(results[spec.name])

print(reg)

## 7. T-stat distributions

In [ ]:
n_specs = len(specs)
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=False)
for ax, spec in zip(axes.flat, specs):
    tstat_hist(results[spec.name].stats, name=spec.name, ax=ax)
fig.suptitle("Per-unit T-stat distributions — kinematics encoding (FDR α=0.05)")
plt.tight_layout()
save_fig(fig, "tstat_histograms_kin", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 8. Pairwise significance structure

Which units are significant across multiple kinematic predictors?

In [ ]:
fig, membership = registry_upset(
    reg,
    entries=reg.names,
    title="Kinematic encoding significance overlap",
)
save_fig(fig, "kin_upset", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

In [ ]:
fig, ax = registry_heatmap(
    reg,
    entries=reg.names,
    title="Pairwise T-stat correlation — kinematic predictors",
)
save_fig(fig, "kin_heatmap", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 9. Example units

In [ ]:
# Show example units for the predictor with most significant units
n_sig = {
    spec.name: results[spec.name].n_sig()["total"]
    for spec in specs
}
top_name = max(n_sig, key=n_sig.get)
print(f"Top predictor by n_sig: {top_name} (n={n_sig[top_name]})")

fig = registry_plot_examples(
    reg, top_name, all_counts_df,
    x_col=results[top_name].spec.predictor_col,
    x_label=results[top_name].spec.predictor_col,
    y_col=results[top_name].spec.response_col,
    n_examples=4, select="top_t",
)
if fig:
    save_fig(fig, f"examples_{top_name}", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

## 10. Session-level RT ~ kinematics correlation

Checks how strongly RT covaries with each kinematic predictor at the
session level. High ρ motivates the partial correlation analysis in eph_03.

In [ ]:
# Session-level Spearman rho between RT and each kinematic predictor.
# Computed on trial-deduplicated data (each row = one trial).
df_sess = (
    all_counts_df
    .drop_duplicates(subset=["session", "trial"])
    .dropna(subset=["reaction_time_firstmove"])
    .query(KIN_QUERY)
    .copy()
)

session_rhos = {}
for key, col in KIN_PREDICTORS.items():
    rows = []
    for sess, g in df_sess.groupby("session"):
        g2 = g.dropna(subset=[col])
        if len(g2) < 10:
            continue
        rho, _ = spearmanr(g2["reaction_time_firstmove"], g2[col])
        rows.append({"session": sess, "rho": rho, "n": len(g2)})
    session_rhos[key] = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for ax, (key, rho_df) in zip(axes.flat, session_rhos.items()):
    ax.hist(rho_df["rho"], bins=15, color=PALETTE["neg"], edgecolor="white")
    ax.axvline(0, color=PALETTE["neutral"], lw=0.8, ls="--")
    ax.set_title(f"RT vs {key}\n(n={len(rho_df)} sessions)")
    ax.set_xlabel("Spearman ρ (session-level)")
    style_ax(ax)

fig.suptitle("Session-level RT ~ kinematics Spearman ρ distribution")
plt.tight_layout()
save_fig(fig, "session_rho_rt_vs_kin", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()